# CodaMOSA vs CoverUp — Experiment Control

This notebook is the single observable control panel. It calls versioned scripts; it does not duplicate the experiment logic.

**Design interpretation:** File B wins. This repository runs CodaMOSA and CoverUp on MT and CM with B1/B2. Pynguin and MuTAP rows come from the collaborator. Full R=10 execution stays locked until the R=2 pilot is complete and explicitly approved.

**Safety:** edit `config/local.env`, which is gitignored. Never paste keys into notebook cells or `experiment.toml`.

**Kernel:** use `.venv (Python 3.14.4)`. This notebook must be opened from inside the repository directory.`

## 0. Kernel and local credentials

Use `.venv (Python 3.14.4)`. Fill `config/local.env` outside this notebook. This cell prints presence only, never values.`

In [ ]:
from pathlib import Path
import sys

# Find the repo root by walking up from the current directory.
_root = Path.cwd()
while _root != _root.parent:
    if (_root / "config" / "experiment.toml").exists():
        break
    _root = _root.parent
ROOT = _root

if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from notebook_control import load_local_env, presence, run, print_status

loaded = load_local_env()
print("Loaded variable names:", loaded)
print("Presence:", presence())
assert all(presence().values()), "Fill config/local.env before API probes."

## 1. No-cost setup and validation

Creates/updates the environment, inventories 588 tasks, builds the 64-run pilot matrix, and runs tests. Safe to rerun.`

In [ ]:
run(["bash", "scripts/setup.sh"], timeout=1200)

## 2. Inspect dataset inventory and pilot plan

Expected totals: 163 MT functions + 425 CM modules = 588. Pilot: 5 MT + 3 disjoint CM × 2 tools × 2 backbones × R=2 = 64 runs.`

In [ ]:
run([sys.executable, "scripts/inventory.py"])
run([sys.executable, "scripts/build_matrix.py", "--pilot"])
run([sys.executable, "scripts/run_experiment.py", "--pilot", "--dry-run"])
print_status()

In [ ]:
run([sys.executable, "scripts/report.py"])
from IPython.display import Markdown
Markdown((ROOT / "experiment_report.md").read_text(encoding="utf-8"))

## 3. Endpoint/tool probes — required before pilot

This validates B1/B2 `model_returned`, CoverUp function calling, and the per-call rate gate. The probe sends `tool_choice: required` (string form) because this endpoint rejects the object form. Both backbones currently route to the same `entah/auto` endpoint (documented File B deviation).`

In [ ]:
probe_run = run([sys.executable, "scripts/probe_endpoints.py"], timeout=300, check=False)
probe = ROOT / "results" / "endpoint_probe.json"
probe_results = __import__("json").loads(probe.read_text(encoding="utf-8"))
for item in probe_results:
    print(item)
failed = [item for item in probe_results if not item.get("passed")]
PROBES_PASSED = not failed
if failed:
    print("\nBLOCKED: a backbone probe did not pass. Fix the failing backbone, rerun this cell, then continue to Section 4.")
    print("Probe report saved to results/endpoint_probe.json")
else:
    print("Both endpoint function-calling probes passed. Continue to pilot smoke test.")
PROBES_PASSED

## 4. Pilot — do not run until Section 3 passes

Runs 64 isolated jobs with checkpoints, 600 s budgets, 5 workers, raw logs, and the USD 900/14-day gates. This is a paid operation.

A smoke run (--max-runs 1) executes first to validate the full tool path before launching the remaining pilot jobs.

**Pause / resume:** The runner checkpoints every completed run to `results/pilot_results.csv`. To pause: `touch results/pause.requested` (or run `python scripts/run_experiment.py --pause`). The current in-flight run will finish, then no new runs launch. To resume: `python scripts/run_experiment.py --resume --retry-failed --pilot`. Re-running without `--retry-failed` skips already-completed runs automatically.`

In [ ]:
PROBES_PASSED = globals().get("PROBES_PASSED", False)
CONFIRM_PAID_PILOT = True

if not PROBES_PASSED:
    print("SKIPPED: Section 3 probe gate has not passed. No pilot task was started.")
elif not CONFIRM_PAID_PILOT:
    print("SKIPPED: set CONFIRM_PAID_PILOT=True after reviewing probe results.")
else:
    # First run one task to observe the full CodaMOSA/CoverUp path before 64 pilot runs.
    try:
        run([sys.executable, "scripts/run_experiment.py", "--pilot", "--max-runs", "1", "--workers", "1"], timeout=2400)
        print_status()
    except RuntimeError as exc:
        print(f"\nSmoke run failed: {exc}")
        print("Aborting remaining pilot runs. Inspect results/generated/ and results/raw/ for the failure.")
        raise
    # If the smoke run is OK, resume all remaining pilot jobs.
    run([sys.executable, "scripts/run_experiment.py", "--pilot"], timeout=172800)

## 5. Pilot monitoring and analysis

Rerun while the pilot is active or after interruption. The runner is resumable — every completed run is checkpointed, so re-running skips what is already done.

**Resume commands:**
- `python scripts/run_experiment.py --resume --pilot` (skips completed runs)
- `python scripts/run_experiment.py --resume --retry-failed --pilot` (also re-runs failed runs)
- `python scripts/run_experiment.py --pause` (pause after current run finishes)`

In [ ]:
print_status()
path = ROOT / "results" / "pilot_results.csv"
if path.exists():
    run([sys.executable, "scripts/analyze.py", str(path), "--output", "results/pilot_analysis"])
    import csv
    from collections import Counter
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    total = len(rows)
    ok = sum(1 for r in rows if r.get("status") == "ok")
    failed = sum(1 for r in rows if r.get("status") == "failed")
    cost = sum(float(r.get("cost_usd") or 0) for r in rows)
    by_cfg = Counter((r.get("suite"), r.get("tool"), r.get("backbone")) for r in rows)
    print(f"\n=== PILOT SUMMARY: {total} runs ===")
    print(f"OK: {ok} | Failed: {failed} | Total cost: ${cost:.6f}")
    print("\nConfiguration breakdown:")
    for (suite, tool, backbone), count in sorted(by_cfg.items()):
        cfg_ok = sum(1 for r in rows if r.get("suite")==suite and r.get("tool")==tool and r.get("backbone")==backbone and r.get("status")=="ok")
        print(f"  {suite}/{tool}/{backbone}: {count} runs, {cfg_ok} OK")
else:
    print("SKIPPED: no pilot_results.csv yet.")

## 6. Full run gate

File B requires pilot timing/cost extrapolation and explicit approval before protocol freeze/full R=10. Approval has been given; this cell will launch the full R=10 run (8,120 jobs) after Section 5 completes.`

In [ ]:
FULL_RUN_APPROVED = True
if not FULL_RUN_APPROVED:
    print("SKIPPED: full R=10 run needs completed pilot analysis and explicit approval.")
    print("Set FULL_RUN_APPROVED = True and rerun this cell when ready.")
else:
    run([sys.executable, "scripts/build_matrix.py"])
    run([sys.executable, "scripts/run_experiment.py"], timeout=1209600)

## 7. Final analysis and figures

Generates Tables IV–IX, paired tests, cost-effectiveness, failure rate, exclusion audit, and Figures 3–5 from real `results.csv`. Runs after the full run completes.`

In [ ]:
results = ROOT / "results" / "results.csv"
if not results.exists():
    print("SKIPPED: no final results.csv yet.")
else:
    run([sys.executable, "scripts/analyze.py", str(results)])
    run([sys.executable, "scripts/figures.py", str(results)])
run([sys.executable, "scripts/report.py"])
print((ROOT / "experiment_report.md").read_text(encoding="utf-8"))